In [ ]:
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

plt.rcParams["figure.figsize"] = (12, 6)  # width, height in inches
set_matplotlib_formats("svg")  # render inline figures as SVG

# Optional tweaks:
plt.rcParams["figure.dpi"] = 96  # effective DPI for other backends
plt.rcParams["savefig.dpi"] = 300  # DPI used when saving figures

import pandas as pd
import seaborn as sns
from pathlib import Path
from matplotlib.ticker import MaxNLocator, MultipleLocator, FuncFormatter, Locator
from IPython.display import clear_output, display
from time import sleep
import requests
from math import floor,ceil
import numpy as np
import json

In [ ]:
sns.set_theme(style='whitegrid')

In [ ]:
def download_curves():
    'download tide curves and metadata'
    r = requests.get("https://gezeiten.bsh.de/data/tides_overview.json")
    r.raise_for_status()
    overview = r.json()
    # print(overview)

    def download(url, file):
        file = Path(file)
        if file.exists():
            return
        r = requests.get(url)
        if r.status_code == 200:
            file.parent.mkdir(parents=True, exist_ok=True)
            if url.endswith('.txt'): r.encoding = "windows-1252"
            tmp = file.with_name(file.name + ".tmp")
            tmp.write_text(r.text)
            tmp.rename(file)
            return True
        # else: print('failed',url)

    for g in overview["gauges"]:
        nr = g["bshnr"]
        print(nr, g["station_name"])
        y = 2026
        download(f"https://gezeiten.bsh.de/data/DE_{nr.rjust(5, '_')}_tides.json", f"bsh/curves/{nr}.json")
        download(f"https://filebox.bsh.de/index.php/s/SbJ3z5NBkpOZloY/download?path=/vb_kurv/deu{y}&files=DE__{nr}{y}.txt", f"bsh/curves/{nr}.txt")
        # download(f"https://filebox.bsh.de/index.php/s/SbJ3z5NBkpOZloY/download?path=/vb_hwnw/deu{y}&files=DE__{nr}{y}.txt", f"bsh/times/{nr}.txt")

download_curves()

In [ ]:
def load_data(file):
    'read tide curve, see https://filebox.bsh.de/index.php/s/SbJ3z5NBkpOZloY/download?path=%2F&files=format_e.txt'
    
    with open(file) as f:
        for l in f:
            if l.startswith('A04'):
                name=l.split('#')[2] 
            if l.startswith('D03'):
                offset=-float(l.split('#')[2].replace(' ',''))
            # if l.startswith('G01'):
            #     MHW=float(l.split('#')[2])+offset
            # if l.startswith('G02'):
            #     MLW=float(l.split('#')[2])+offset
            if l.startswith('LLL'):
                break
                
        df=pd.read_csv(f,sep='#',skiprows=26,header=None)
                
    # df=pd.read_csv(file,sep='#',skiprows=26,header=None)
    df.columns='k nr moon event dow date time height qual day tz cycle transit jd x'.split()
    df=df.dropna(subset=['height'])
    df.height+=offset
    df['datetime']=pd.to_datetime((df['date']+'_'+df['time']).str.replace(' ','0'),format='%d.%m.%Y_%H:%M')
    del df['x']

    t,lw,hw=None,None,None
    T,LW,HW=[],[],[]
    MOON,moon=[],None
    TMOON,tmoon=[],None
    for i,r in df.iterrows():
        # print(i,r)
        if r.event=='N': 
            try:
                lw=r.height
                f=df[(df.datetime>r.datetime) & (df.event=='H')].iloc[0]
                t=f['datetime']
                hw=f['height']
            except:
                t=None
        if r.event=='H': 
            try: lw=df[(df.datetime>r.datetime) & (df.event=='N')].iloc[0]['height']
            except: lw=None
        T.append(t)
        LW.append(lw)
        HW.append(hw)
        if r.moon.strip(): tmoon=r.datetime
        TMOON.append(tmoon)
        moon=r.moon.strip() or moon
        MOON.append(int(moon) if moon else None)
    df['T']=T
    df['T']=(df.datetime-df['T']).dt.total_seconds()/3600
    df['HW']=HW
    df['LW']=LW
    df['range']=df.HW-df.LW
    df['moon']=MOON
    df['mage']=TMOON
    df['mage']=(df.datetime-df.mage).dt.total_seconds()/3600/24
    df=df.dropna(subset=['T','HW','LW','moon'])
    df['h']=(df.height-df.LW)/(df.HW-df.LW)
    # df['age']=df.moon.apply(lambda v: 'neaps' if v % 2 else 'springs')
    spring_delay=1
    df['age']=df.apply(lambda r:('neaps' if r.moon % 2 else 'springs') if spring_delay<=r.mage<spring_delay+4 else 'mid',axis=1)

    return df

In [ ]:
def get_meta(file):
    'read tide gauge metadata'
    with open(file) as f:
        m=json.load(f)
    # print(r.text)
    d=m['years'][0]['2026']
    del d['hwnw_prediction']
    del m['years']
    d.update(m)
    return d

# get_meta('506P')    

In [ ]:
df=load_data('bsh/curves/506P.txt')
meta=get_meta('bsh/curves/506P.json')
df

In [ ]:
sns.histplot(df[df.event!='K'],x='range',hue='age');
df[df.event!='K'].groupby('age',as_index=0).range.mean()

In [ ]:
sns.histplot(df[df.event!='K'],x='height',hue='age',bins=80);

In [ ]:
sns.lineplot(df,x='T',y='h',hue='age',errorbar='pi');

In [ ]:
class ThresholdLocator(Locator):
    """
    Combine two locators, switching at a specified threshold.

    Parameters
    ----------
    low_locator : matplotlib.ticker.Locator
        Locator used for tick positions at or below the threshold.
    high_locator : matplotlib.ticker.Locator
        Locator used for tick positions above the threshold.
    threshold : float
        Data coordinate where the switch occurs.
    include_threshold_in : {"low", "high", "both"}
        Determines which side gets the exact threshold tick (default "both").
    """
    def __init__(self, low_locator, high_locator, threshold):
        super().__init__()
        self.low_locator = low_locator
        self.high_locator = high_locator
        self.threshold = threshold

    def set_axis(self, axis):
        super().set_axis(axis)
        if hasattr(self.low_locator, "set_axis"):
            self.low_locator.set_axis(axis)
        if hasattr(self.high_locator, "set_axis"):
            self.high_locator.set_axis(axis)

    def tick_values(self, vmin, vmax):
        # normalize the interval
        lo, hi = sorted((vmin, vmax))
        thr = self.threshold
        ticks = []

        if lo <= thr:
            upper = thr
            ticks_low = self.low_locator.tick_values(lo, thr)
            ticks_low = list(filter(lambda v:v<thr,ticks_low))
            ticks.extend(ticks_low)

        if hi >= thr:
            lower = thr
            ticks_high = self.high_locator.tick_values(thr, hi)
            ticks_high = list(filter(lambda v:v>=thr,ticks_high))
            ticks.extend(ticks_high)

        ticks = np.unique(ticks)
        return ticks[(ticks >= lo) & (ticks <= hi)]

    def __call__(self):
        vmin, vmax = self.axis.get_view_interval()
        return self.tick_values(vmin, vmax)

In [ ]:
def make_curves(file,save=0):
    file=Path(file)
    # print(file.stem)
    df=load_data(file)
    meta=get_meta(file.with_name(file.stem+'.json'))
    name=meta['bshnr']+' '+meta['station_name']
    print(name)
    try:
        MRS=meta['MSpTh']/100
        MRN=meta['MNpTh']/100
        offset=meta['SKN (ueber PNP)']/100
        MHWS=meta['MSpHW']/100-offset
        MHWN=meta['MNpHW']/100-offset
        MLWS=meta['MSpNW']/100-offset
        MLWN=meta['MNpNW']/100-offset
        shift=ceil(MHWS)*2
    except:
        MRS,MRN,MHWS,MHWN,MLWS,MLWN=[float('nan')]*6
        MLWS=0
        shift=3*2
        
    d=df.groupby(["age", "T"], as_index=False)["h"].mean()
    d['h']=d.groupby('age')['h'].transform(lambda s: s.rolling(11, center=True).mean())
    tmin,tmax=floor(d['T'].min()),ceil(d['T'].max())
    shift-=floor(d['T'].min())
    
    def xformat(val, pos):
        # print(val,pos)
        if val==0: return "HW"
        # if abs(val)==6: return "LW"
        if val<tmin: 
            v=(val+shift)/2
            if v==int(v): return f'{v:.0f}m'
            return None
        return f'{val:.0f}h'
        
    def yformat(val, pos):
        if val==0 or val==1: return None
        return round(val,1)
    
    fig, ax = plt.subplots(figsize=(11,5))
    
    d[d.age=='springs'].plot(x='T',y='h',color='r',style='-',label=f'Springs {MRS:.1f}m',ax=ax,zorder=2,linewidth=2)
    d[d.age=='neaps'].plot(x='T',y='h',color='b',style='--',label=f'Neaps {MRN:.1f}m',ax=ax,zorder=1,linewidth=2)
    
    plt.plot([2*MHWS-shift]*2,[1,0.97],'r',linewidth=2)
    plt.plot([2*MHWN-shift]*2,[1,0.97],'b',linewidth=2)
    plt.plot([2*MLWS-shift]*2,[0,0.03],'r',linewidth=2)
    plt.plot([2*MLWN-shift]*2,[0,0.03],'b',linewidth=2)
    
    plt.plot([2*MLWS-shift,2*MHWS-shift],[0,1],'r',linewidth=0.3)
    plt.plot([2*MLWN-shift,2*MHWN-shift],[0,1],'b',linewidth=0.3)
    
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.xaxis.set_major_formatter(FuncFormatter(xformat))
    ax.yaxis.set_major_formatter(FuncFormatter(yformat))
    ax.tick_params(
        axis="both", which="major",bottom=True, top=True, #left=True,
        labeltop=True, labelbottom=True,labelleft=0,labelright=1,
        length=10, width=2, colors="green", direction="in" 
    )
    ax.grid(which="major", linestyle='-', linewidth=0.8, color="black", alpha=0.5)
    
    # ax.xaxis.set_minor_locator(MultipleLocator(1/6))
    ax.xaxis.set_minor_locator(ThresholdLocator(MultipleLocator(0.2),MultipleLocator(1/6),tmin),)
    ax.yaxis.set_minor_locator(MultipleLocator(0.025))
    ax.tick_params(
        axis="both", which="minor",bottom=True, top=True, #left=True,
        length=5, width=1, colors="black", direction="in" 
    )
    ax.tick_params(axis="y", which='both', direction="inout")
    ax.grid(which="minor", linewidth=0.4, color="black", alpha=0.3)
    
    for spine in ("left", "right", "top", "bottom"):
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(1)
    
    ax.spines["right"].set_position(("data", 0))
    plt.ylim(0,1)
    plt.xlim(2*floor(MLWS)-shift,tmax)
    # plt.legend()
    plt.xlabel(None)
    plt.title(name)
    # plt.show()
    plt.tight_layout()
    if save: plt.savefig(f'bsh/pdf/{name}.{save}')

make_curves('bsh/curves/101P.txt','pdf')    

In [ ]:
d=Path('bsh/curves')
for f in d.glob('*.txt'):
    fig=make_curves(f,'pdf')
    plt.close(fig) 

In [485]:
!pdfunite bsh/pdf/*.pdf tidecurves.pdf